# Regex-only AIUB Faculty Scraper (Name / Email / Room No)

**NLP mini project** — collects the faculty **name, e-mail and room number** of the
*Faculty of Science & Technology* from
`https://www.aiub.edu/faculty-list/faculty-of-science--technology` using **regular expressions only**.

### Rules followed in this notebook

| Requirement | How it is handled |
|---|---|
| Only regular expressions | Every extraction, filter and cleanup is an `re` pattern. No BeautifulSoup, no `lxml`, no `json` module, no `pandas`. |
| Collect **name**, **email**, **room no** | Regex 6 pulls those keys out of the raw text; Regex 8-21 clean, validate and analyse them. |
| Small and runnable | Standard library only: `urllib`, `re`, `csv`, `collections`, `os`. |

### How the site is built (the interesting part)

The page renders its faculty cards with **JavaScript**, so the HTML the server returns
contains no names and no rooms. The notebook therefore:

1. downloads the page HTML and **proves** the data is not in it,
2. regex-matches the `<script>` file behind the page,
3. regex-matches the data-feed URL out of that script,
4. downloads the feed as **raw text** and parses every record with regex,
5. cleans the three fields, then writes `output/fst_faculty.csv`.

### Steps 10-13 add on top of the requirement

| Feature | What it gives you |
|---|---|
| `collect()` for any faculty | the same pipeline across all five faculties, plus a per-faculty summary CSV |
| `find(pattern)` | a regex lookup over name, e-mail, department, room, position and interests |
| profile links | the `faculty-profile?q=...` URL that the site itself builds, one per row |
| research-interest analysis | tokenising, stop-words, term counts, and a check on why word counts and phrase counts disagree |
| room and department map | normalised building + room keys, rooms holding more than one person, department sizes |

Run all cells top-to-bottom (Restart & Run All) — 13 steps, about 15 seconds, 3 CSVs.

In [1]:
# ---------------------------------------------------------------
# Standard library only.
# ---------------------------------------------------------------
import re
import csv
import os
import collections
import urllib.request

PAGE_URL = ("https://www.aiub.edu/faculty-list/faculty-of-science--technology"
            "?faculty=FACULTY+OF+SCIENCE+%26+TECHNOLOGY")
BASE_URL = "https://www.aiub.edu"
OUT_DIR = "output"

os.makedirs(OUT_DIR, exist_ok=True)
print("notebook folder:", os.getcwd())

notebook folder: /Users/roybipro/Developer/JNBook/NLP/Faculty-Data


## Step 1 — Download the page and check what is actually in the HTML

In [2]:
def fetch(url):
    # Download a URL and return the body as plain text. Nothing is parsed here.
    request = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=60) as response:
        return response.read().decode("utf-8", "replace")

# Regex 1: anything that looks like an e-mail address
LOOKS_LIKE_EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w-]+(?:\.[\w-]+)*")
# Regex 2: the word 'room' / 'Room No' anywhere in the document
ROOM_WORD = re.compile(r"room\s*(?:no|number|#)?\.?", re.I)

html = fetch(PAGE_URL)

print("HTML characters       :", len(html))
print("e-mail looking hits   :", LOOKS_LIKE_EMAIL.findall(html))
print("'room' hits in HTML   :", len(ROOM_WORD.findall(html)))
print("occurrences of 'RoomNo':", html.count("RoomNo"))

HTML characters       : 123174
e-mail looking hits   : ['arahman@aiub.edu', 'arahman@aiub.edu', 'info@aiub.edu']
'room' hits in HTML   : 0
occurrences of 'RoomNo': 0


Confirmed: the served HTML is only the site shell plus menu — **no room numbers at all**.
The cards are injected by JavaScript, so the next job is to find *where* that script loads
its data. Everything below is discovered with regex; nothing is hard-coded.

## Step 2 — Find the data feed with regex

In [3]:
# Regex 3: the src attribute of the employee-profile script tag
#   matches:  src="/Content/employee-profile/script.js?v=23"
SCRIPT_SRC = re.compile(r'src="(?P<src>[^"]*employee-profile/script\.js[^"]*)"')

# Regex 4: the URL inside the fetch("...") call of that script
#   matches:  fetch("/Files/Uploads/public-employee-profiles/employeeProfiles.json?v=1.0.11")
FETCH_CALL = re.compile(r'fetch\( *"(?P<url>/Files/Uploads/[^"]+\.json[^"]*)"')

script_path = SCRIPT_SRC.search(html).group("src")
script_text = fetch(BASE_URL + script_path)
feed_path = FETCH_CALL.search(script_text).group("url")
FEED_URL = BASE_URL + feed_path

print("page script :", script_path)
print("data feed   :", FEED_URL)

page script : /Content/employee-profile/script.js?v=23
data feed   : https://www.aiub.edu/Files/Uploads/public-employee-profiles/employeeProfiles.json?v=1.0.11


## Step 3 — Download the feed and keep it as raw text

The text is also snapshotted to `output/faculty_feed_raw.txt` so the input data can be
inspected or handed in with the report. Each run re-downloads a fresh copy.

In [4]:
raw_text = fetch(FEED_URL)

# Snapshot of the raw input (not used for parsing - parsing always reads raw_text)
RAW_SNAPSHOT = os.path.join(OUT_DIR, "faculty_feed_raw.txt")
with open(RAW_SNAPSHOT, "w", encoding="utf-8") as handle:
    handle.write(raw_text)

print("feed characters :", len(raw_text))
print("snapshot saved  :", os.path.abspath(RAW_SNAPSHOT))
print("first 300 chars:")
print(raw_text[:300])

feed characters : 294863
snapshot saved  : /Users/roybipro/Developer/JNBook/NLP/Faculty-Data/output/faculty_feed_raw.txt
first 300 chars:
{"EmployeeProfileLightList":[{"IsShowProfileDetails":true,"CvPersonal":{"ID":7256,"UserId":60989,"Name":"MD. ANWARUL KABIR","Email":"kabir@aiub.edu"},"Faculty":"FACULTY OF SCIENCE & TECHNOLOGY","Designation":"Faculty","Position":"ASSOCIATE PROFESSOR","PositionID":137,"HrDepartment":"DEPARTMENT OF CO


## Step 4 — Split the text into one record per employee

The feed arrives as one very long line. Every employee object starts with the same key, so
a **lookahead split** cuts the text right in front of each of those keys while keeping the
opening brace inside the record:

```python
RECORD_START = re.compile(r'(?=\{"IsShowProfileDetails")')
```

Two details worth remembering for a viva answer:

* `\{` escapes the brace, otherwise `{...}` would be read as a repeat operator.
* `(?=...)` is a zero-width lookahead: it marks the cut position but does not consume the
  brace, so each piece still begins with `{"IsShowProfileDetails"`.

After that the parser is one tiny pattern per field — `"Key":"value"` — where `[^"]*`
means "everything up to the next quote".

In [5]:
# Regex 5: the split point, i.e. the position just before every record
RECORD_START = re.compile(r'(?=\{"IsShowProfileDetails")')

records = [r for r in RECORD_START.split(raw_text)
           if r.startswith('{"IsShowProfileDetails')]

def field(record, key):
    # Regex 6:  "RoomNo":"817"   ->   817
    match = re.search('"' + key + '":"(?P<value>[^"]*)"', record)
    return match.group("value") if match else ""

KEYS = ("Name", "Email", "Faculty", "Designation", "Position",
        "HrDepartment", "RoomNo", "BuildingNo",
        "ResearchInterests", "AcademicInterests")

people = [{key.lower(): field(record, key) for key in KEYS} for record in records]

print("records found  :", len(records))
print("employees      :", len(people))
print("sample         :", people[0])
print("missing name   :", sum(1 for p in people if not p["name"]))
print("missing email  :", sum(1 for p in people if not p["email"]))
print("empty room no  :", sum(1 for p in people if not p["roomno"]))

records found  : 446
employees      : 446
sample         : {'name': 'MD. ANWARUL KABIR', 'email': 'kabir@aiub.edu', 'faculty': 'FACULTY OF SCIENCE & TECHNOLOGY', 'designation': 'Faculty', 'position': 'ASSOCIATE PROFESSOR', 'hrdepartment': 'DEPARTMENT OF COMPUTER SCIENCE', 'roomno': 'DNO -216', 'buildingno': 'D - Building', 'researchinterests': 'Digital Transformation', 'academicinterests': 'Big Data,Model-driven software engineering,Software Quality Assurance,Software Testing,Software Architectures'}
missing name   : 0
missing email  : 0
empty room no  : 38


## Step 5 — Keep only the Faculty of Science & Technology

The stored value is `FACULTY OF SCIENCE & TECHNOLOGY`, while the page URL spells the same
thing `FACULTY+OF+SCIENCE+%26+TECHNOLOGY`, and other AIUB pages write `Science and
Technology`. One regex covers all of those spellings and any letter case.

In [6]:
# Regex 7: 'SCIENCE & TECHNOLOGY' in any spelling of the connector
IN_SCIENCE_TECH = re.compile(r"SCIENCE\s*(?:&amp;|&|AND)\s*TECHNOLOGY", re.I)

fst = [p for p in people if IN_SCIENCE_TECH.search(p["faculty"])]

print("all faculties on the feed:")
for name, count in collections.Counter(p["faculty"] for p in people).most_common():
    print("   {:>3}  {}".format(count, name))
print()
print("records selected for this project:", len(fst))

all faculties on the feed:
   196  FACULTY OF SCIENCE & TECHNOLOGY
    97  FACULTY OF ENGINEERING
    72  FACULTY OF ARTS AND SOCIAL SCIENCES
    65  FACULTY OF BUSINESS ADMINISTRATION
    16  FACULTY OF HEALTH AND LIFE SCIENCES

records selected for this project: 196


## Step 6 — Clean the three target fields (the NLP part)

| Field | Problem in the raw data | Regex fix |
|---|---|---|
| name | `PROF. DR. KH. ABDUL MALEQUE` is ALL CAPS and carries honorifics | Regex 8 strips known titles, Regex 12 title-cases the rest. `Md.` is *kept* because in Bangladeshi names it belongs to the given name |
| email | the loose unanchored Regex 1 matches any address-shaped fragment, including ones inside links | Regex 9 is anchored and strict, so it can be used as a real validation check |
| room no | messy formats: `817`, `DNGA05`, `DNO -216`, `9210K`, `1122 (OR)` | Regex 10 splits block letters + digits + wing letter, Regex 11 captures bracketed notes separately |
| department / building / position | stored in ALL CAPS as `DEPARTMENT OF COMPUTER SCIENCE` | Regex 13 title-cases them but keeps connectors like `of` / `and` lowercase |
| research interests | one uneven blob: `" Materials and Processing,Polymer Chemistry"` | Regex 15 splits on commas and semicolons, Regex 13 tidies each phrase |
| profile link | the feed stores no URL at all | Regex 14 takes the part before `@`, which is exactly how the site builds `faculty-profile?q=...` |

In [7]:
# Regex 8: a leading honorific token such as 'PROF.' or 'DR.'
LEAD_TITLE = re.compile(r"^(?P<title>[A-Za-z]+\.)\s+(?P<rest>.*)$")
KNOWN_TITLES = {"prof", "dr", "mr", "ms", "mst", "engr", "rev"}

# Regex 9: a strictly valid e-mail address, anchored at both ends
VALID_EMAIL = re.compile(r"^[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)*"
                         r"\.[A-Za-z]{2,}$")

# Regex 10: room code = optional block letters + digits + optional wing letter
#   'DNO -216' -> block=DNO room=216 | '9210K' -> room=9210 wing=K
ROOM_PARTS = re.compile(r"(?P<block>[A-Za-z]{1,6})?[\s\-]*(?P<room>\d{1,6})"
                        r"(?:[\s\-]*(?P<wing>[A-Za-z]))?", re.I)

# Regex 11: a bracketed note such as '(OR)' or '(on Study Leave)'
BRACKET_NOTE = re.compile(r"\( *[^)]* *\)")

def clean_name(text):
    titles = []
    while True:
        match = LEAD_TITLE.match(text)
        if not match or match.group("title").rstrip(".").lower() not in KNOWN_TITLES:
            break
        titles.append(match.group("title"))
        text = match.group("rest")
    # Regex 12: capitalise the first letter of every word
    body = re.sub(r"\b([a-z])", lambda m: m.group(1).upper(), text.strip().lower())
    return (" ".join(t.title() for t in titles) + " " + body).strip()

def clean_room(text):
    match = ROOM_PARTS.search(text or "")
    note = BRACKET_NOTE.search(text or "")
    if not match:
        return {"block": "", "room": "", "wing": "", "note": ""}
    return {"block": (match.group("block") or "").upper(),
            "room": match.group("room"),
            "wing": (match.group("wing") or "").upper(),
            "note": note.group(0) if note else ""}

# Regex 13: lower-case connector words inside a label ('Department Of X' -> 'Department of X')
SMALL_WORDS = re.compile(r"(?<= )\b(Of|And|The|For|In|On)\b")

def clean_label(text):
    return SMALL_WORDS.sub(lambda m: m.group(1).lower(),
                           re.sub(r"\s+", " ", text).strip().title())

# Regex 14: the user-id in front of the '@'. The site builds its own profile links
#           from exactly this piece: /faculty-list/faculty-profile?q=kabir
USER_ID = re.compile(r"(?P<user>[^@]+)@")
PROFILE_LINK = BASE_URL + "/faculty-list/faculty-profile?q="

# Regex 15: interests arrive as one blob separated by commas or semicolons
INTEREST_SPLIT = re.compile(r"[,;]+")

def clean_interests(text):
    parts = [re.sub(r"\s+", " ", x).strip(" .") for x in INTEREST_SPLIT.split(text or "")]
    return ", ".join(x for x in parts if x)

def make_row(p):
    match = USER_ID.match(p["email"].strip().lower())
    room = clean_room(p["roomno"])
    return {
        "name": clean_name(p["name"]),
        "email": p["email"].strip().lower(),
        "room_no": p["roomno"].strip(),
        "room_parsed": room["block"] + room["room"] + room["wing"],
        "building": clean_label(p["buildingno"]),
        "position": clean_label(p["position"]),
        "department": clean_label(p["hrdepartment"]),
        "interests": clean_interests(p["researchinterests"] or p["academicinterests"]),
        "profile_url": PROFILE_LINK + (match.group("user") if match else ""),
    }

rows = [make_row(p) for p in fst]

print("cleaned rows:", len(rows))
print()
for a, b in [("MD. ANWARUL KABIR", "name"), ("PROF. DR. KH. ABDUL MALEQUE", "name"),
             ("DR. MD. ABDULLAH - AL - JUBAIR", "name"), ("DNO -216", "room"),
             ("9210K", "room"), ("1122 (OR)", "room"), ("", "room")]:
    out = clean_name(a) if b == "name" else clean_room(a)
    print("   {!r:<36} -> {!r}".format(a, out))

cleaned rows: 196

   'MD. ANWARUL KABIR'                  -> 'Md. Anwarul Kabir'
   'PROF. DR. KH. ABDUL MALEQUE'        -> 'Prof. Dr. Kh. Abdul Maleque'
   'DR. MD. ABDULLAH - AL - JUBAIR'     -> 'Dr. Md. Abdullah - Al - Jubair'
   'DNO -216'                           -> {'block': 'DNO', 'room': '216', 'wing': '', 'note': ''}
   '9210K'                              -> {'block': '', 'room': '9210', 'wing': 'K', 'note': ''}
   '1122 (OR)'                          -> {'block': '', 'room': '1122', 'wing': '', 'note': '(OR)'}
   ''                                   -> {'block': '', 'room': '', 'wing': '', 'note': ''}


## Step 7 — Sanity checks before saving

In [8]:
bad_emails = [r["email"] for r in rows if not VALID_EMAIL.match(r["email"])]
no_room = [r["name"] for r in rows if not r["room_no"]]
dupes = [e for e, c in collections.Counter(r["email"] for r in rows).items() if c > 1]

# Regex 16: a profile link is only trusted if it matches the shape the site uses
VALID_LINK = re.compile(r"^https://www\.aiub\.edu/faculty-list/faculty-profile\?q=[a-z0-9._-]+$")
bad_links = [r["profile_url"] for r in rows if not VALID_LINK.match(r["profile_url"])]
with_interest = [r for r in rows if r["interests"]]

print("rows                        :", len(rows))
print("e-mails failing the pattern :", len(bad_emails), bad_emails)
print("duplicate e-mails           :", len(dupes), dupes)
print("rows without a room number  :", len(no_room), no_room)
print("room codes parsed           :",
      sum(1 for r in rows if r["room_parsed"]), "/", len(rows))
print("profile links built         :", len(rows) - len(bad_links), "/", len(rows),
      "| malformed:", bad_links)
print("rows carrying interest text :", len(with_interest), "/", len(rows))

rows                        : 196
e-mails failing the pattern : 0 []
duplicate e-mails           : 0 []
rows without a room number  : 11 ['Prof. Dr. Dip Nandi', 'Shahrin Chowdhury', 'Md. Asiful Islam', 'Asma Fariha', 'Fahmida Alam', 'Dr. M M Manjurul Islam', 'Md Moman Ul Haque Khan', 'Tamanna Zaman Bristy', 'Kamrun Naher Koli', 'Sadman Rafi', 'Dr. Mousumi Bala']
room codes parsed           : 183 / 196
profile links built         : 196 / 196 | malformed: []
rows carrying interest text : 185 / 196


## Step 8 — Save to CSV and preview

In [9]:
CSV_PATH = os.path.join(OUT_DIR, "fst_faculty.csv")
COLUMNS = ("name", "email", "room_no", "room_parsed", "building",
           "position", "department", "interests", "profile_url")

with open(CSV_PATH, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=COLUMNS)
    writer.writeheader()
    writer.writerows(rows)

print("saved ->", os.path.abspath(CSV_PATH))
print("columns:", ", ".join(COLUMNS))
print()
print("{:<31} {:<27} {:<9} {}".format("NAME", "EMAIL", "ROOM NO", "PROFILE LINK"))
print("-" * 100)
for r in rows[:12]:
    print("{:<31} {:<27} {:<9} {}".format(r["name"][:31], r["email"][:27],
                                          r["room_no"] or "-", r["profile_url"]))
print("... and {} more rows in the CSV".format(max(0, len(rows) - 12)))

saved -> /Users/roybipro/Developer/JNBook/NLP/Faculty-Data/output/fst_faculty.csv
columns: name, email, room_no, room_parsed, building, position, department, interests, profile_url

NAME                            EMAIL                       ROOM NO   PROFILE LINK
----------------------------------------------------------------------------------------------------
Md. Anwarul Kabir               kabir@aiub.edu              DNO -216  https://www.aiub.edu/faculty-list/faculty-profile?q=kabir
Mashiour Rahman                 mashiour@aiub.edu           DNGA05    https://www.aiub.edu/faculty-list/faculty-profile?q=mashiour
Salma Begom                     salma@aiub.edu              817       https://www.aiub.edu/faculty-list/faculty-profile?q=salma
Dr. Mohammed Jashim Uddin       drjashim@aiub.edu           1122      https://www.aiub.edu/faculty-list/faculty-profile?q=drjashim
Prof. Dr. Kh. Abdul Maleque     maleque@aiub.edu            DN0915    https://www.aiub.edu/faculty-list/faculty-prof

## Step 9 — Small text analysis over the collected data

Note: bracketed leave notes are stripped first (Regex 11), so `LECTURER (on Study Leave)`
is counted as `LECTURER` — otherwise the same job would appear as several different titles.

In [10]:
# Positions, ignoring bracketed leave notes (Regex 11 reused)
positions = collections.Counter(BRACKET_NOTE.sub("", r["position"]).strip().title()
                                for r in rows)

# Letter blocks that prefix a room code, e.g. DN / DNO / DNGA in DN0915, DNO -216
blocks = collections.Counter(
    m.group(0) for r in rows
    for m in [re.match(r"[A-Za-z]+", r["room_parsed"])] if m)

# Most common name words, from a regex word tokenizer, minus stop-word-ish tokens
stop = {"md", "dr", "prof", "mst", "khan", "miah", "begum", "binte", "ibn", "ahmed"}
words = collections.Counter(
    w.capitalize() for r in rows for w in re.findall(r"[A-Za-z']{3,}", r["name"])
    if w.lower() not in stop)

# How many user-ids are a single lowercase word (i.e. basically the first name)?
short_ids = sum(1 for r in rows if re.match(r"^[a-z]+$", r["email"].split("@")[0]))

print("FST faculty collected  :", len(rows))
print("with a room number     :", len(rows) - len(no_room))
print("e-mail domains         :", collections.Counter(
      r["email"].split("@")[1] for r in rows).most_common(3))
print("single-word user ids   :", short_ids, "/", len(rows))
print("positions              :", positions.most_common())
print("room blocks            :", blocks.most_common(6))
print("top name words         :", words.most_common(10))

FST faculty collected  : 196
with a room number     : 185
e-mail domains         : [('aiub.edu', 196)]
single-word user ids   : 104 / 196
positions              : [('Lecturer', 81), ('Assistant Professor', 67), ('Associate Professor', 26), ('Professor', 17), ('Senior Assistant Professor', 3), ('Senior Associate Professor', 1), ('Senior Lecturer', 1)]
room blocks            : [('DN', 98), ('DS', 11), ('DNGA', 9), ('D', 5), ('DNO', 3), ('DE', 3)]
top name words         : [('Islam', 19), ('Hasan', 12), ('Rahman', 8), ('Mohammad', 7), ('Alam', 6), ('Rifat', 5), ('Hossain', 5), ('Uddin', 4), ('Farzana', 4), ('Jahan', 4)]


## Step 10 — The same pipeline for any faculty

Nothing in the code is tied to Science & Technology. Grouping the feed by the value each record
declares gives every faculty on site, and `make_row()` works for all of them, so `collect()`
returns the same table for whichever faculty you ask for.

The menu links are read with a regex too, which is how you can check that the `?faculty=` value
in the page URL really is the same string the feed stores.

In [11]:
# Regex 17: the faculty a record declares for itself
FACULTY_OF = re.compile(r'"Faculty":"([^"]*)"')

# Regex 18: the faculty pages linked from the menu of the page we opened
FACULTY_PAGE = re.compile(r'href="(/faculty-list/[a-z0-9\-]+)"')

by_faculty = collections.defaultdict(list)
faculty_of = [(FACULTY_OF.search(r).group(1) if FACULTY_OF.search(r) else "(none)")
              for r in records]
for name, p in zip(faculty_of, people):
    by_faculty[name].append(p)

# the value re-read from the raw record must equal the one Regex 6 stored earlier
disagree = sum(1 for a, b in zip(faculty_of, people) if a != b["faculty"])

pages = sorted(set(FACULTY_PAGE.findall(html)))

def collect(faculty_name):
    return [make_row(p) for p in by_faculty[faculty_name]]

print("faculties in the feed       :", len(by_faculty))
print("raw-vs-parsed disagreements :", disagree)
print("faculty pages linked in menu:", len(pages))
for page in pages:
    print("   ", page)
print()
print("{:>6}  {:<40} {:>6} {:>7}".format("RECORDS", "FACULTY", "ROOMS", "COVERAGE"))
for name in sorted(by_faculty, key=lambda n: -len(by_faculty[n])):
    recs = by_faculty[name]
    rooms = sum(1 for p in recs if p["roomno"])
    print("{:>6}  {:<40} {:>6} {:>7}".format(
        len(recs), name.title(), rooms, str(round(100 * rooms / len(recs))) + "%"))

# a different faculty, exactly the same code path
eng = collect("FACULTY OF ENGINEERING")
print()
print("Engineering through the same pipeline: {} rows".format(len(eng)))
for r in eng[:3]:
    print("   {:<30} {:<25} {}".format(r["name"], r["email"], r["room_no"] or "-"))

faculties in the feed       : 5
raw-vs-parsed disagreements : 0
faculty pages linked in menu: 6
    /faculty-list/faculties
    /faculty-list/faculty-of-arts-and-social-sciences
    /faculty-list/faculty-of-business-administration
    /faculty-list/faculty-of-engineering
    /faculty-list/faculty-of-health-and-life-science
    /faculty-list/faculty-of-science--technology

RECORDS  FACULTY                                   ROOMS COVERAGE
   196  Faculty Of Science & Technology             185     94%
    97  Faculty Of Engineering                       90     93%
    72  Faculty Of Arts And Social Sciences          63     88%
    65  Faculty Of Business Administration           57     88%
    16  Faculty Of Health And Life Sciences          13     81%

Engineering through the same pipeline: 97 rows
   Prof. M. A. Quaiyum            maquaiyum@aiub.edu        -
   Dr. Shahriyar Masud Rizvi      shahriyar@aiub.edu        417
   Dr. Md. Hasan Imam             hasan.imam@aiub.edu       D416


Only aggregate counts are saved for the faculties outside this project's scope. The feed covers
all 446 employees, while the assignment asks for Science & Technology, so the repo keeps the
196 detailed rows plus one summary line per faculty — no extra names or addresses.

In [12]:
SUMMARY_PATH = os.path.join(OUT_DIR, "faculty_summary.csv")

with open(SUMMARY_PATH, "w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["faculty", "records", "with_room_no",
                     "with_interest_text", "room_coverage_pct"])
    for name in sorted(by_faculty, key=lambda n: -len(by_faculty[n])):
        recs = by_faculty[name]
        rooms = sum(1 for p in recs if p["roomno"])
        interests = sum(1 for p in recs
                        if (p["researchinterests"] or p["academicinterests"]).strip())
        writer.writerow([name.title(), len(recs), rooms, interests,
                         round(100 * rooms / len(recs))])

# the two coverage columns look identical - check whether they are the same people
missing_room = {p["name"] for p in people if not p["roomno"]}
missing_text = {p["name"] for p in people
                if not (p["researchinterests"] or p["academicinterests"]).strip()}
print("records with no room number   :", len(missing_room))
print("records with no interest text :", len(missing_text))
print("exactly the same people       :", missing_room == missing_text)
print("(an empty PersonalOtherInfo block leaves both fields blank)")
print()
print("saved ->", os.path.abspath(SUMMARY_PATH))
print()
with open(SUMMARY_PATH, encoding="utf-8") as handle:
    print(handle.read())

records with no room number   : 38
records with no interest text : 38
exactly the same people       : True
(an empty PersonalOtherInfo block leaves both fields blank)

saved -> /Users/roybipro/Developer/JNBook/NLP/Faculty-Data/output/faculty_summary.csv

faculty,records,with_room_no,with_interest_text,room_coverage_pct
Faculty Of Science & Technology,196,185,185,94
Faculty Of Engineering,97,90,90,93
Faculty Of Arts And Social Sciences,72,63,63,88
Faculty Of Business Administration,65,57,57,88
Faculty Of Health And Life Sciences,16,13,13,81



## Step 11 — A regex lookup over what was collected

`find()` compiles whatever pattern you hand it and searches the chosen columns
case-insensitively. That is a two-line search index made from the same tool the
collector already uses.

In [13]:
SEARCH_FIELDS = ("name", "email", "department", "room_no", "position", "interests")

def find(pattern, fields=SEARCH_FIELDS):
    # Regex 19: built at call time, so any pattern works
    rx = re.compile(pattern, re.I)
    return [r for r in rows if any(rx.search(r[field]) for field in fields)]

for query in (r"machine learning", r"^DN", r"data\s+science", r"physics",
              r"study leave", r"dr\."):
    print("{:<18} {:>3} hits".format(query, len(find(query))))

print()
print("who works on machine learning?")
for r in find(r"machine learning", ("interests",))[:6]:
    print("   {:<30} {:<27} room {}".format(r["name"][:30], r["department"][:27],
                                            r["room_no"] or "-"))

print()
print("rooms in the DNGA block:")
for r in find(r"^DNGA", ("room_no",)):
    print("   {:<30} {:<27} {}".format(r["name"][:30], r["department"][:27], r["room_no"]))

machine learning     1 hits
^DN                110 hits
data\s+science       1 hits
physics             21 hits
study leave         20 hits
dr\.                63 hits

who works on machine learning?
   Al Jubair Hossain              Department of Computer Scie room 9310F

rooms in the DNGA block:
   Mashiour Rahman                Department of Computer Scie DNGA05
   Dr. Abhijit Bhowmik            Department of Computer Scie DNGA11
   Sharfuddin Mahmood             Department of Computer Scie DNGA12
   Dr. Akinul Islam Jony          Department of Computer Scie DNGA03
   Victor Stany Rozario           Department of Computer Scie DNGA02
   Mohaimen- Bin- Noor            Department of Computer Scie DNGA01
   Dr. Md. Abdullah - Al - Jubair Department of Computer Scie DNGA04
   Tanvir Ahmed                   Department of Computer Scie DNGA09
   Rifath Mahmud                  Department of Computer Scie DNGA07


## Step 12 — Research interests: tokenising, stop-words and term counts

This is the closest thing in the project to real NLP work. 185 of the 196 records carry free
text, comma separated and unevenly spaced (`" Materials and Processing,Polymer Chemistry"`), so
it gets tokenised, stop-worded and counted two ways: by total mentions and by how many distinct
faculty use the term.

In [14]:
# Regex 20: a term starts with a letter and may keep the symbols that survive inside
#           real research phrases - C++, .NET, IoT, deep-learning, e-commerce
TOKEN = re.compile(r"[A-Za-z][A-Za-z0-9+#.\-/]{2,}")
STOP = set("and or the of in on at for with to from a an using used based system "
           "science sciences engineering technology field fields area areas their "
           "this that various etc including as is are be well".split())

terms = collections.Counter()
faculty_using = collections.Counter()
per_person = []

for r in rows:
    if not r["interests"]:
        continue
    found = [t.lower().strip(".-/") for t in TOKEN.findall(r["interests"])]
    found = [t for t in found if len(t) > 2 and t not in STOP]
    per_person.append(len(found))
    terms.update(found)
    faculty_using.update(set(found))

print("faculty with interest text  :", len(per_person), "/", len(rows))
print("terms written per faculty   : avg {:.1f}, max {}".format(
      sum(per_person) / len(per_person), max(per_person)))
print("distinct terms after stop-wording:", len(terms))
print()
print("top 10 by total mentions :", [t for t, _ in terms.most_common(10)])
print("top 10 by how many faculty use them:")
for term, n in faculty_using.most_common(10):
    print("   {:<20} {} faculty".format(term, n))
print()
for topic in ("machine learning", "deep learning", "data science", "security",
              "internet of things", "image processing", "cloud"):
    print("   topic {:<19} {:>3} faculty".format(
          topic, len(find(re.escape(topic), ("interests",)))))

# why a word count and a phrase count disagree - 'machine' is mostly 'Machine to Machine'
m2m = len(find(r"machine\s+to\s+machine", ("interests",)))
ml = len(find(r"machine\s+learning", ("interests",)))
word = len(find(r"\bmachine\b", ("interests",)))
print()
print("the same text, counted two ways:")
print("   records mentioning the word 'machine'      :", word)
print("   ... in the phrase 'machine to machine'      :", m2m)
print("   ... in the phrase 'machine learning'        :", ml)
print("   -> a bare word list would overstate machine-learning interest by",
      word - ml, "records")

TERMS_PATH = os.path.join(OUT_DIR, "research_terms.csv")
with open(TERMS_PATH, "w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["term", "mentions", "faculty_using_it"])
    for term, n in faculty_using.most_common(60):
        writer.writerow([term, terms[term], n])
print()
print("saved ->", os.path.abspath(TERMS_PATH), "(top 60 terms)")

faculty with interest text  : 185 / 196
terms written per faculty   : avg 9.2, max 52
distinct terms after stop-wording: 166

top 10 by total mentions : ['intelligence', 'learning', 'deep', 'processing', 'data', 'software', 'artificial', 'computer', 'machine', 'natural']
top 10 by how many faculty use them:
   learning             76 faculty
   deep                 76 faculty
   intelligence         75 faculty
   artificial           65 faculty
   processing           53 faculty
   data                 46 faculty
   computer             40 faculty
   natural              38 faculty
   vision               36 faculty
   things               34 faculty

   topic machine learning      1 faculty
   topic deep learning        76 faculty
   topic data science          1 faculty
   topic security             15 faculty
   topic internet of things   34 faculty
   topic image processing     25 faculty
   topic cloud                 5 faculty

the same text, counted two ways:
   records mentioni

## Step 13 — Room clashes and a department directory

Two questions you can only answer once the codes are normalised: does any room hold more than
one person, and how big is each department? Room `817` in the D building and `817` somewhere
else are different places, so the comparison key is building + code with everything that is not
a letter or digit removed.

In [15]:
# Regex 21: comparable room key = building + code, punctuation and spaces stripped
NON_ALNUM = re.compile(r"[^A-Z0-9]")

def room_key(r):
    return NON_ALNUM.sub("", r["building"] + "-" + r["room_parsed"])

in_room = collections.defaultdict(list)
for r in rows:
    if r["room_parsed"]:
        in_room[room_key(r)].append(r["name"])

shared = {k: v for k, v in in_room.items() if len(v) > 1}
by_dept = collections.Counter(r["department"] for r in rows)

print("people with a room         :", sum(1 for r in rows if r["room_parsed"]))
print("distinct room keys         :", len(in_room))
print("keys holding more than one :", len(shared))
for key, names in sorted(shared.items())[:6]:
    print("   {:<14} {}".format(key, ", ".join(names)))

print()
print("departments in this faculty:", len(by_dept))
for dept, n in by_dept.most_common(8):
    print("   {:>3}  {}".format(n, dept))
print()
small = sorted(d for d, n in by_dept.items() if n == 1)
print("single-person departments  :", len(small), small[:4])

people with a room         : 183
distinct room keys         : 166
keys holding more than one : 14
   A11122         Dr. Mohammed Jashim Uddin, A.G.M. Zaman, Dr. Mohammad Mahmudul Hasan, Dr. Muhammad Firoz Mridha
   A66101         Dr. Md. Saef Ullah Miah, Md. Tanzeem Rahat
   DBDN0124       Zahiduddin Ahmed, Israt Jahan Mouri
   DBDN0126       Shakila Rahman, Samiha Islam Abrita
   DBDN0127A      Rifat Ibn Alam, Md. Rifat
   DBDN0217       Kawser Irom Rushee, Dr. Mohammad Rabiul Islam

departments in this faculty: 4
   140  Department of Computer Science
    27  Department of Mathematics
    21  Department of Natural Science [Physics]
     8  Department of Natural Science [Chemistry]

single-person departments  : 0 []


## Notes, limits and next steps

**What the project now does** — one download, 21 numbered regular expressions, no third-party
library. It collects the three required fields (name, e-mail, room number) for the Faculty of
Science & Technology, then adds normalised room codes, cleaned names and labels, profile links,
the research-interest text, a `find()` lookup over any column, the same pipeline for every other
faculty, and three CSVs: detail rows, a per-faculty summary, and a research-term list.

**Known limits (worth stating in a report)**

* The parser assumes the feed keeps the `{"IsShowProfileDetails"` record prefix and the current
  key names. If AIUB changes the feed, Regex 5 and 6 need re-checking — that fragility is the
  price of regex-only parsing versus a JSON/HTML parser.
* A value containing an escaped quote (`\"`) would break `[^"]*`; none exist in the feed today.
* 11 records have no room number at the source, and codes such as `C` or `VUES` carry no digits
  so they cannot be normalised — 183 of 196 parse. Blanks are kept, never invented.
* Unigram counts mislead on their own: the word `machine` mostly comes from *Machine to
  Machine*, not *Machine Learning*. Step 12 prints both counts so the disagreement is visible
  instead of hidden.
* Room sharing is judged on `BuildingNo` + code, so two people in one physical room still look
  different if the site spells their building differently (`D - Building` vs `D BUILDING - ANNEX`).
* The profile link reuses the pattern the site builds itself (`?q=` + the part before `@`),
  checked against the live page for a lower-case and a mixed-case user-id, but it depends on
  that convention holding.
* `interests` falls back to academic interests when research interests are empty, because the
  site leaves one of the two blank for some records.

**Next ideas**

* Bigram counting: `re.findall(r"\b[a-z]+ [a-z]+\b", interests.lower())`, then drop phrases made
  of two stop-words — that fixes the `machine` / `machine learning` confusion properly.
* Turn `find()` into a small command-line lookup, or write the room map out as a floor list per
  building.
* Snapshot the feed weekly and regex-diff the name list to catch joiners and leavers.